## Inference visualisation

Load `vis_data.pkl` from any training run and render the full figure suite.
Set **`VIS_DATA_PATH`** in the load cell, then **Run All**.

In [ ]:
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import spearmanr

from cxval.vis import STYLE, add_colorbar
from cxval.analysis import (
    pairwise_decode, crosscontext_decode, generalisation_matrix,
    value_decode_within, value_decode_cross, value_gen_matrix,
    binary_value_decode_within, binary_value_decode_cross,
    value_decode_within_partialled, value_decode_cross_partialled,
    binary_value_decode_within_partialled, binary_value_decode_cross_partialled,
    compute_neural_rdm, compute_model_rdms, rsa_compare,
    held_out_stim_decode, cross_group_decode, cross_group_binary_decode,
    decoder_axis_similarity, plot_generalisation_heatmap, filter_act_dict,
    compute_unit_tuning, selectivity_index, preferred_value_proportions,
    time_resolved_decode, policy_similarity_analysis,
)
plt.style.use(STYLE)

In [ ]:
# ── set path to vis_data.pkl ──────────────────────────────────────────
VIS_DATA_PATH = Path("../results/RNN_15s2c_h64_g0.9_ep1_bptt10_upd10_seed42_20260513_233127/vis_data.pkl")

with open(VIS_DATA_PATH, "rb") as f:
    vd = pickle.load(f)

print(f"Loaded: {VIS_DATA_PATH}")

# ── figure save directory ────────────────────────────────────────────────
FIGURES_BASE = Path("../figures/13_05_2026_nmlg_lab_meeting")
run_id   = VIS_DATA_PATH.parent.name
save_dir = FIGURES_BASE / run_id
save_dir.mkdir(parents=True, exist_ok=True)
print(f"run_id  : {run_id}")
print(f"figures → {save_dir}")

In [ ]:
value_matrix          = vd["value_matrix"]
n_stimuli             = vd["n_stimuli"]
n_contexts            = vd["n_contexts"]
stimuli               = vd["stimuli"]
contexts              = vd["contexts"]
stim_group_info       = vd["stim_group_info"]
stim_timesteps        = vd["stim_timesteps"]
reward_timesteps      = vd["reward_timesteps"]
reward_lick           = vd["reward_lick"]
reward_lick_fa      = vd["reward_lick_fa"]
reward_no_lick        = vd["reward_no_lick"]
lick_cost             = vd["lick_cost"]
infer_trial_data      = vd["infer_trial_data"]
infer_action_seq      = vd["infer_action_seq"]
infer_trial_structure = vd["infer_trial_structure"]

# derived inference arrays
inf_stim   = np.array([d["stimulus"]         for d in infer_trial_data])
inf_ctx    = np.array([d["context"]          for d in infer_trial_data])
inf_lick   = np.array([d["licked"]           for d in infer_trial_data])
inf_lkct   = np.array([d["lick_count"]       for d in infer_trial_data])
inf_vest   = np.array([d["value_estimate"]   for d in infer_trial_data])
inf_ravail = np.array([d["reward_available"] for d in infer_trial_data]).astype(bool)

def _actual_r(licked, ravail):
    if licked:
        base  = reward_lick if ravail else reward_lick_fa
        extra = lick_cost if (not ravail and lick_cost != 0) else 0.0
        return base + extra
    return reward_no_lick

inf_tderr = np.array([
    _actual_r(l, ra) for l, ra in zip(inf_lick.tolist(), inf_ravail.tolist())
]) - inf_vest

# per-stim per-context aggregates
val_sc  = np.full((n_stimuli, n_contexts), np.nan)
lick_sc = np.full((n_stimuli, n_contexts), np.nan)
lkrt_sc = np.full((n_stimuli, n_contexts), np.nan)
td_sc   = np.full((n_stimuli, n_contexts), np.nan)
for si in range(n_stimuli):
    for ci in range(n_contexts):
        m = (inf_stim == si) & (inf_ctx == ci)
        if m.sum() == 0:
            continue
        val_sc[si, ci]  = inf_vest[m].mean()
        lick_sc[si, ci] = inf_lick[m].mean()
        lkrt_sc[si, ci] = inf_lkct[m].mean() / reward_timesteps
        td_sc[si, ci]   = inf_tderr[m].mean()

# colour / marker palettes
_CTX_PALETTE = ["#5C6BC0", "#FFA726", "#66BB6A", "#EF5350", "#26C6DA"]
ctx_colors   = [_CTX_PALETTE[ci % len(_CTX_PALETTE)] for ci in range(n_contexts)]
stim_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ctx_markers = ["o", "s", "^", "D", "v"]

x = np.arange(n_stimuli)
w = min(0.8 / n_contexts, 0.4)

# period lick rates from raw action sequence
def _lick_rate(s, e):
    seg = [infer_action_seq[t] == 0 for t in range(s, e)]
    return float(np.mean(seg)) if seg else np.nan

_iti_rates, _stim_rates, _rew_r_rates, _rew_u_rates = [], [], [], []
_pc_ctx, _pc_stim, _pc_ravail = [], [], []

for trial in infer_trial_structure:
    iti_s,  iti_e  = trial["iti_window"]
    stim_s, stim_e = trial["stim_window"]
    rew_s,  rew_e  = trial["reward_window"]
    rr = _lick_rate(rew_s, rew_e)
    _iti_rates.append(_lick_rate(iti_s, iti_e))
    _stim_rates.append(_lick_rate(stim_s, stim_e))
    _rew_r_rates.append(rr if trial["reward_available"] else np.nan)
    _rew_u_rates.append(rr if not trial["reward_available"] else np.nan)
    _pc_ctx.append(trial["context"])
    _pc_stim.append(trial["stimulus"])
    _pc_ravail.append(trial["reward_available"])

_pc_ctx    = np.array(_pc_ctx)
_pc_stim   = np.array(_pc_stim)
_pc_ravail = np.array(_pc_ravail, dtype=bool)

def _per_stim_ctx(rates):
    rates = np.array(rates, dtype=float)
    out = np.full((n_stimuli, n_contexts), np.nan)
    for si in range(n_stimuli):
        for ci in range(n_contexts):
            m = (_pc_stim == si) & (_pc_ctx == ci)
            vals = rates[m]
            if np.any(~np.isnan(vals)):
                out[si, ci] = np.nanmean(vals)
    return out

_iti_sc  = _per_stim_ctx(_iti_rates)
_stim_sc = _per_stim_ctx(_stim_rates)
_rewr_sc = _per_stim_ctx(_rew_r_rates)
_rewu_sc = _per_stim_ctx(_rew_u_rates)

print(f"n_stimuli={n_stimuli}  n_contexts={n_contexts}  infer_trials={len(infer_trial_data)}")
print(f"stimulus groups: {[g['name'] for g in stim_group_info]}")

In [ ]:
def add_stim_group_labels(ax, x_pos, bar_half_w=0.45, y_line=-0.13, y_text=-0.22, fontsize=7.5):
    # Draw group bracket lines + text labels below the x-axis.
    # Uses ax.get_xaxis_transform(): x = data coords, y = axes fraction.
    # Single-member groups get label only; multi-member groups get bracket + label.
    trans = ax.get_xaxis_transform()
    for g in stim_group_info:
        idxs = g["indices"]
        if not idxs:
            continue
        xs  = [x_pos[i] for i in idxs]
        lo, hi = min(xs) - bar_half_w, max(xs) + bar_half_w
        mid    = (lo + hi) / 2

        if len(idxs) > 1:
            ax.plot([lo, hi], [y_line, y_line], transform=trans,
                    color="0.35", lw=0.9, clip_on=False, solid_capstyle="butt")
            for xv in [lo, hi]:
                ax.plot([xv, xv], [y_line, y_line - 0.018], transform=trans,
                        color="0.35", lw=0.9, clip_on=False)

        ax.text(mid, y_text, g["short"], transform=trans,
                ha="center", va="top", fontsize=fontsize, color="0.25", clip_on=False)

## Figure 0 — Trial-by-trial behaviour

In [ ]:
# ── Figure 0: trial-by-trial behaviour ──────────────────────────────────────
# Set last_n_reps=None to show all training; set e.g. last_n_reps=3 for final reps only.
last_n_reps = None   # ← edit here

infer_trial_data = vd["infer_trial_data"]
all_trial_data   = vd.get("all_trial_data", None)
reward_timesteps = vd["reward_timesteps"]
n_trials         = vd.get("n_trials", None)
context_reps     = vd.get("context_reps", None)
trials_per_phase = vd.get("trials_per_phase", None)

# ── build inference array ─────────────────────────────────────────────────────
infer_td = np.array([
    (i, d["context"], d["stimulus"],
     d["reward_available"], d["licked"], d["value_estimate"], d["lick_count"])
    for i, d in enumerate(infer_trial_data)
], dtype=float)
# cols: 0=trial_i 1=context 2=stimulus 3=reward_avail 4=licked 5=value 6=lick_count

ctx_colors  = [plt.cm.Set2(v) for v in np.linspace(0, 0.75, max(n_contexts, 1))]
stim_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

def _behaviour_fig(data_td, title_suffix, ax_title_prefix="", smooth_w=5):
    """Three-panel behaviour plot: value estimate, lick prob, lick rate."""
    n_t = int(data_td[:, 0].max()) + 1

    # context background spans
    ctx_arr = data_td[:, 1].astype(int)
    chg     = np.where(np.diff(ctx_arr) != 0)[0] + 1
    bstarts = np.concatenate([[0], chg])
    bends   = np.concatenate([chg, [len(ctx_arr)]])
    bctxs   = ctx_arr[bstarts]

    fig, axes = plt.subplots(3, 1, figsize=(14, 8),
                             gridspec_kw={"height_ratios": [1, 1, 1], "hspace": 0.45})

    for bs, be, ctx in zip(bstarts, bends, bctxs):
        for ax in axes:
            ax.axvspan(bs - 0.5, be - 0.5, alpha=0.15,
                       color=ctx_colors[ctx], zorder=0, linewidth=0)

    for si in range(n_stimuli):
        c  = stim_colors[si % len(stim_colors)]
        m  = data_td[:, 2] == si
        xi = data_td[m, 0].astype(int)
        v  = data_td[m, 5]
        lp = data_td[m, 4]
        lr = data_td[m, 6] / reward_timesteps

        def _smooth(y):
            if smooth_w > 1 and len(y) >= smooth_w:
                s = np.convolve(y, np.ones(smooth_w) / smooth_w, mode="valid")
                return xi[smooth_w - 1:], s
            return xi, y

        xx, sv  = _smooth(v)
        xx, slp = _smooth(lp)
        xx, slr = _smooth(lr)
        axes[0].plot(xx, sv,  color=c, lw=1.3, label=stimuli[si])
        axes[1].plot(xx, slp, color=c, lw=1.3)
        axes[2].plot(xx, slr, color=c, lw=1.3)

    # ground-truth dotted lines on value panel
    for si in range(n_stimuli):
        c = stim_colors[si % len(stim_colors)]
        for ci in range(n_contexts):
            axes[0].axhline(value_matrix[si, ci], color=c, lw=0.8, linestyle=":", alpha=0.5)

    axes[0].set_ylabel("Value estimate")
    axes[0].set_title(f"{ax_title_prefix}Value estimate  (dotted = GT per context)",
                      fontsize=10)
    axes[0].legend(ncol=n_stimuli, fontsize=8, loc="upper left")

    axes[1].set_ylabel("Lick prob\n(1st reward step)")
    axes[1].set_ylim(-0.02, 1.02)
    axes[1].set_title(f"{ax_title_prefix}Lick probability at first reward-window timestep",
                      fontsize=10)
    axes[1].axhline(0.5, color="gray", lw=0.7, linestyle="--", alpha=0.5)

    axes[2].set_ylabel(f"Lick rate\n(all {reward_timesteps} rew. steps)")
    axes[2].set_ylim(-0.02, 1.02)
    axes[2].set_xlabel("Trial index")
    axes[2].set_title(f"{ax_title_prefix}Mean lick rate across reward window",
                      fontsize=10)
    axes[2].axhline(0.5, color="gray", lw=0.7, linestyle="--", alpha=0.5)

    if n_contexts > 1:
        ctx_handles = [Patch(facecolor=ctx_colors[c], alpha=0.6, label=contexts[c])
                       for c in range(n_contexts)]
        axes[0].legend(
            handles=[Line2D([0],[0], color=stim_colors[si % len(stim_colors)],
                            label=stimuli[si]) for si in range(n_stimuli)] + ctx_handles,
            ncol=n_stimuli + n_contexts, fontsize=8, loc="upper left",
        )

    fig.suptitle(f"{run_id}  ·  {title_suffix}", fontsize=9, y=1.01)
    plt.tight_layout()
    return fig

# ── Figure 0a: inference (plasticity off) ────────────────────────────────────
fig0a = _behaviour_fig(infer_td, title_suffix="inference  [plasticity off]",
                       ax_title_prefix="Inference — ")
fig0a.savefig(save_dir / "fig0a_inference_behaviour.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Figure 0b: training behaviour ────────────────────────────────────────────
if all_trial_data is not None:
    train_td = np.array([
        (i, d["context"], d["stimulus"],
         d["reward_available"], d["licked"], d["value_estimate"], d["lick_count"])
        for i, d in enumerate(all_trial_data)
    ], dtype=float)

    if last_n_reps is not None and trials_per_phase is not None:
        cutoff   = len(all_trial_data) - last_n_reps * n_contexts * trials_per_phase
        cutoff   = max(0, cutoff)
        train_td = train_td[int(cutoff):]
        train_td[:, 0] -= train_td[0, 0]   # re-index from 0
        suffix = f"training  (last {last_n_reps} rep{'s' if last_n_reps != 1 else ''} per context)"
    else:
        suffix = "training  (full)"

    fig0b = _behaviour_fig(train_td, title_suffix=suffix,
                           ax_title_prefix="Training — ")
    fig0b.savefig(save_dir / "fig0b_training_behaviour.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("all_trial_data not found in vis_data.pkl — re-run the training notebook to enable Fig 0b.")

## Figure 1 — Lick–value calibration and performance

In [ ]:
# ── Figure 1: Lick–value calibration and performance summary ────────────────
show_point_labels = False   # True → annotate each scatter point with sX,cY

lp_flat, rp_flat, si_flat, ci_flat = [], [], [], []
for si in range(n_stimuli):
    for ci in range(n_contexts):
        m = (inf_stim == si) & (inf_ctx == ci)
        if m.sum() == 0:
            continue
        lp_flat.append(float(inf_lick[m].mean()))
        rp_flat.append(float(value_matrix[si, ci]))
        si_flat.append(si); ci_flat.append(ci)

lp_flat = np.array(lp_flat); rp_flat = np.array(rp_flat)
r_sp = spearmanr(rp_flat, lp_flat)[0] if len(lp_flat) > 1 else np.nan

pct_consumed = inf_lick[inf_ravail].mean()  * 100 if  inf_ravail.any() else np.nan
pct_fa       = inf_lick[~inf_ravail].mean() * 100 if (~inf_ravail).any() else np.nan

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

# -- calibration scatter --
ax = axes[0]
for lp, rp, si, ci in zip(lp_flat, rp_flat, si_flat, ci_flat):
    ax.scatter(rp, lp,
               color=stim_colors[si % len(stim_colors)],
               marker=ctx_markers[ci % len(ctx_markers)],
               s=90, zorder=3)
    if show_point_labels:
        ax.annotate(f"{stimuli[si]},{contexts[ci]}", (rp, lp),
                    textcoords="offset points", xytext=(5, 3), fontsize=6)

ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Reward probability")
ax.set_ylabel("Lick probability\n(1st reward-window timestep)")
ax.set_title(
    f"Lick–value calibration  (Spearman r = {r_sp:.3f})\n"
    f"lick probability sampled at 1st reward-window timestep  [plasticity off]"
)

# -- performance summary --
ax = axes[1]
bars = ax.bar(["Reward\nconsumed (%)", "False alarm\nrate (%)"],
              [pct_consumed, pct_fa], color=["steelblue", "salmon"], width=0.45)
ax.set_ylim(0, 110); ax.set_ylabel("%")
ax.set_title("Performance summary\n[inference / plasticity off]")
for bar, val in zip(bars, [pct_consumed, pct_fa]):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 1,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
fig.savefig(save_dir / "fig1_lick_value_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 1b — Policy Similarity Analysis (PSA)

Measures how close the model's policy is to the optimal policy for each context.

Stimuli are classified per context as **high** (reward prob ≥ 1−threshold), **low**
(≤ threshold), or **mid** (between).  The PSA score is `1 − MAE` between actual lick
probabilities and the desired policy (high=1, low=0), optionally clipped to account for
any softmax saturation limits.  **Mid-stimulus lick probability** is the most informative
secondary metric: a well-calibrated model should lick at a rate proportional to the mid
stimulus's reward probability.

In [ ]:
# ── Figure 1b: PSA — Policy Similarity Analysis ──────────────────────────
# clip_lo / clip_hi: set to the empirically observed min/max lick prob if the
# network's softmax never fully saturates.  Leave at 0/1 for standard MAE.
_psa_threshold = 0.25
_psa_clip_lo   = 0.0
_psa_clip_hi   = 1.0

psa_results = policy_similarity_analysis(
    lick_sc, value_matrix,
    threshold=_psa_threshold,
    clip_lo=_psa_clip_lo,
    clip_hi=_psa_clip_hi,
)

# ── print summary ─────────────────────────────────────────────────────────
print(f"PSA threshold={_psa_threshold}  (high≥{1 - _psa_threshold:.2f}, low≤{_psa_threshold})\n")
for ci in range(n_contexts):
    r = psa_results[ci]
    print(f"  {contexts[ci]}:"
          f"  high={r['high_lick']:.3f}  low={r['low_lick']:.3f}"
          f"  mid={r['mid_lick']:.3f}"
          f"  PSA score={r['psa_score']:.3f}  delta={r['psa_delta']:.3f}")

# ── figure ────────────────────────────────────────────────────────────────
_grp_colors = {"high": "#DD8452", "mid": "#8C8C8C", "low": "#4C72B0"}
_ctx_col     = [plt.cm.Set2(v) for v in np.linspace(0, 0.75, max(n_contexts, 1))]

has_mid = any(len(psa_results[ci]["mid_stim"]) > 0 for ci in range(n_contexts))
n_panels = 3 if has_mid else 2
fig, axes = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 4.5))
if n_panels == 2:
    axes = list(axes) + [None]

# ── Panel 1: grouped bars — high / mid / low mean lick prob per context ───
ax = axes[0]
x_ctx  = np.arange(n_contexts)
n_grps = 3
width  = 0.22
for i, (key, col) in enumerate(_grp_colors.items()):
    vals = [psa_results[ci][f"{key}_lick"] for ci in range(n_contexts)]
    bars = ax.bar(x_ctx + (i - 1) * width, vals, width=width,
                  color=col, alpha=0.85, label=key, zorder=2)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)

ax.axhline(_psa_clip_hi, color="black", lw=0.9, linestyle="--", alpha=0.55,
           label=f"desired high ({_psa_clip_hi})")
ax.axhline(_psa_clip_lo, color="black", lw=0.9, linestyle=":",  alpha=0.55,
           label=f"desired low ({_psa_clip_lo})")
ax.set_xticks(x_ctx)
ax.set_xticklabels(contexts)
ax.set_ylim(-0.08, 1.22)
ax.set_ylabel("Mean lick probability")
ax.set_title(
    f"Policy by value group\n"
    f"(high ≥ {1-_psa_threshold:.2f},  low ≤ {_psa_threshold})"
)
ax.legend(fontsize=8)

# ── Panel 2: PSA score and delta per context ──────────────────────────────
ax = axes[1]
psa_scores = [psa_results[ci]["psa_score"] for ci in range(n_contexts)]
psa_deltas = [psa_results[ci]["psa_delta"] for ci in range(n_contexts)]
w2 = 0.35
bars_s = ax.bar(x_ctx - w2 / 2, psa_scores, width=w2,
                color="#55A868", alpha=0.85, label="PSA score  (1 − MAE)", zorder=2)
bars_d = ax.bar(x_ctx + w2 / 2, psa_deltas, width=w2,
                color="#C44E52", alpha=0.85, label="PSA delta  (high − low)", zorder=2)
ax.axhline(1.0, color="gray", lw=0.8, linestyle="--", alpha=0.5)
ax.axhline(0.0, color="gray", lw=0.8, linestyle=":",  alpha=0.5)
ax.set_xticks(x_ctx)
ax.set_xticklabels(contexts)
ax.set_ylim(-0.08, 1.22)
ax.set_ylabel("Score")
ax.set_title("PSA score and policy range\nper context")
ax.legend(fontsize=8)
for bars in [bars_s, bars_d]:
    for bar in bars:
        val = bar.get_height()
        if not np.isnan(val):
            yoff = 0.02 if val >= 0 else -0.08
            va   = "bottom" if val >= 0 else "top"
            ax.text(bar.get_x() + bar.get_width() / 2, val + yoff,
                    f"{val:.2f}", ha="center", va=va, fontsize=8)

# ── Panel 3: mid-stimulus lick prob vs reward prob ────────────────────────
if has_mid and axes[2] is not None:
    ax = axes[2]
    _ctx_markers = ["o", "s", "^", "D", "v"]
    for ci in range(n_contexts):
        for si in psa_results[ci]["mid_stim"]:
            lp = lick_sc[si, ci]
            rp = value_matrix[si, ci]
            ax.scatter(rp, lp,
                       color=_ctx_col[ci],
                       marker=_ctx_markers[ci % len(_ctx_markers)],
                       s=90, zorder=3)
            ax.annotate(stimuli[si], (rp, lp),
                        textcoords="offset points", xytext=(5, 3), fontsize=8)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5, label="ideal")
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("Reward probability")
    ax.set_ylabel("Lick probability")
    ax.set_title("Mid-stimulus: lick prob vs value\n(excluded from PSA score)")
    ctx_handles = [
        plt.Line2D([0], [0], marker=_ctx_markers[ci % len(_ctx_markers)],
                   color="w", markerfacecolor=_ctx_col[ci], markersize=9,
                   label=contexts[ci])
        for ci in range(n_contexts)
    ]
    ax.legend(handles=ctx_handles + [plt.Line2D([0],[0], color="k", lw=1,
              linestyle="--", label="ideal")], fontsize=8)

fig.suptitle(
    f"PSA — Policy Similarity Analysis  [inference, plasticity off]\n"
    f"low ≤ {_psa_threshold}  ·  mid  ·  high ≥ {1-_psa_threshold:.2f}  "
    f"·  clip [{_psa_clip_lo}, {_psa_clip_hi}]",
    y=1.03,
)
plt.tight_layout()
fig.savefig(save_dir / "fig1b_psa.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 2 — Per-stimulus per-context inference summary

In [ ]:
# ── Figure 2: per-stimulus per-context inference summary ─────────────────
has_groups = len(stim_group_info) > 1
bot_margin = 0.17 if has_groups else 0.08

row_data = [
    (val_sc,  "Value estimate\n(mean over reward window)",
     "Value estimates  [mean over all reward-window timesteps]"),
    (lick_sc, "Lick probability\n(1st reward-window step)",
     "Lick probability at 1st reward-window timestep"),
    (lkrt_sc, f"Outcome lick proportion\n(mean licks / {reward_timesteps} steps)",
     f"Outcome lick proportion  [all {reward_timesteps} reward-window timesteps]"),
]

fig, axes = plt.subplots(3, 1, figsize=(max(8, 2 + 1.6 * n_stimuli), 10), sharex=True,
                         gridspec_kw={"hspace": 0.55})

for ax, (data, ylabel, title) in zip(axes, row_data):
    for ci in range(n_contexts):
        offset = (ci - (n_contexts - 1) / 2) * w
        ax.bar(x + offset, data[:, ci], width=w,
               color=ctx_colors[ci], alpha=0.85, label=contexts[ci], zorder=2)
        for si in range(n_stimuli):
            ax.plot([x[si] + offset - w / 2, x[si] + offset + w / 2],
                    [value_matrix[si, ci]] * 2, color="black", lw=2.0, zorder=5)
    ax.set_ylim(-0.1, 1.1)
    ax.axhline(0, color="gray", lw=0.5, linestyle=":")
    ax.axhline(1, color="gray", lw=0.5, linestyle=":")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title + "\n(black lines = ground-truth reward probability)", fontsize=9)
    ax.legend(fontsize=8)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(stimuli)
axes[-1].set_xlabel("Stimulus" if not has_groups else "")

plt.tight_layout(rect=[0, bot_margin, 1, 1])
if has_groups:
    add_stim_group_labels(axes[-1], x)

fig.suptitle("Per-stimulus per-context inference summary  [plasticity off]", y=1.01)
fig.savefig(save_dir / "fig2_per_stim_inference_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 3 — Period-specific lick rates (overview)

In [ ]:
# ── Figure 3: period-specific lick rates ───────────────────────────────
overall = {
    "ITI":                  np.nanmean(_iti_rates),
    "Stim":                 np.nanmean(_stim_rates),
    "Reward\n(rewarded)":   np.nanmean(_rew_r_rates),
    "Reward\n(unrewarded)": np.nanmean(_rew_u_rates),
}

fig, (ax_ov, ax_ctx) = plt.subplots(1, 2, figsize=(11, 4.5))

bars = ax_ov.bar(list(overall), list(overall.values()),
                 color=["lightgray", "#4C72B0", "#55A868", "#DD8452"], width=0.55)
ax_ov.axhline(0.5, color="gray", linestyle="--", lw=0.8, alpha=0.6)
ax_ov.set_ylim(0, 1.05); ax_ov.set_ylabel("Mean lick rate")
ax_ov.set_title("Overall lick rate by trial period\n[inference / plasticity off]")
for bar, val in zip(bars, overall.values()):
    if not np.isnan(val):
        ax_ov.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                   f"{val:.2f}", ha="center", va="bottom", fontsize=10)

_xp = np.arange(4)
for ci in range(n_contexts):
    m = _pc_ctx == ci
    ctx_ov = [
        np.nanmean(np.array(_iti_rates,   float)[m]),
        np.nanmean(np.array(_stim_rates,  float)[m]),
        np.nanmean(np.array(_rew_r_rates, float)[_pc_ravail & m]),
        np.nanmean(np.array(_rew_u_rates, float)[~_pc_ravail & m]),
    ]
    offset = (ci - (n_contexts - 1) / 2) * 0.3
    ax_ctx.bar(_xp + offset, ctx_ov, width=0.28,
               color=ctx_colors[ci], alpha=0.85, label=contexts[ci])

ax_ctx.set_xticks(_xp)
ax_ctx.set_xticklabels(["ITI", "Stim", "Reward\n(rewarded)", "Reward\n(unrewarded)"])
ax_ctx.axhline(0.5, color="gray", linestyle="--", lw=0.8, alpha=0.6)
ax_ctx.set_ylim(0, 1.05); ax_ctx.set_ylabel("Mean lick rate")
ax_ctx.set_title("Lick rate by period and context\n[inference / plasticity off]")
ax_ctx.legend(fontsize=9)

fig.suptitle("Period-specific lick rates — check for always-licking policy", y=1.02)
plt.tight_layout()
fig.savefig(save_dir / "fig3_period_lick_rates_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 4 — Per-stimulus lick rates by trial period

In [ ]:
# ── Figure 4: per-stimulus per-context lick rates by trial period ──────────
has_groups = len(stim_group_info) > 1
bot_margin = 0.15 if has_groups else 0.06

_period_rows = [
    (_iti_sc,  "ITI lick rate\n(all ITI timesteps)",
     "ITI period  —  should be ~0 if policy is selective"),
    (_stim_sc, "Stim lick rate\n(all stim timesteps)",
     f"Stimulus period  [all {stim_timesteps} timesteps]  —  should be ~0 if not pre-licking"),
    (_rewr_sc, "Reward lick rate\n(rewarded trials)",
     f"Reward window — rewarded trials  [{reward_timesteps} timesteps]"),
    (_rewu_sc, "Reward lick rate\n(unrewarded trials)",
     f"Reward window — unrewarded trials  [{reward_timesteps} timesteps]"),
]

fig, axes = plt.subplots(4, 1, figsize=(max(8, 2 + 1.6 * n_stimuli), 13), sharex=True,
                         gridspec_kw={"hspace": 0.55})

for ax, (data, ylabel, title) in zip(axes, _period_rows):
    for ci in range(n_contexts):
        offset = (ci - (n_contexts - 1) / 2) * w
        ax.bar(x + offset, data[:, ci], width=w,
               color=ctx_colors[ci], alpha=0.85, label=contexts[ci], zorder=2)
        for si in range(n_stimuli):
            ax.plot([x[si] + offset - w / 2, x[si] + offset + w / 2],
                    [value_matrix[si, ci]] * 2, color="black", lw=2.0, zorder=5)
    ax.set_ylim(-0.05, 1.1)
    ax.axhline(0, color="gray", lw=0.5, linestyle=":")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title + "\n(black lines = GT reward probability)", fontsize=9)
    ax.legend(fontsize=8)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(stimuli)
axes[-1].set_xlabel("Stimulus" if not has_groups else "")

plt.tight_layout(rect=[0, bot_margin, 1, 1])
if has_groups:
    add_stim_group_labels(axes[-1], x)

fig.suptitle("Per-stimulus lick rates across all trial periods  [inference / plasticity off]",
             y=1.01)
fig.savefig(save_dir / "fig4_period_lick_rates_per_stim.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 5 — Decoding generalisation (inference activations)

In [ ]:
# ── load decoding data ────────────────────────────────────────────────────
infer_activations = vd.get("infer_activations")
stim_groups       = vd.get("stim_groups", {"all": None})
group_colors      = vd.get("group_colors", {"all": "black"})
pooling           = vd.get("pooling", "average")
n_folds           = vd.get("n_folds", 5)
cell_size         = 1.1

if infer_activations is None:
    raise KeyError(
        "vis_data.pkl does not contain 'infer_activations'. "
        "Re-run the training notebook — the save cell now includes activations."
    )

# ── Figure 5: stim identity + Ridge value generalisation ─────────────────
infer_stim_acc       = pairwise_decode(
    infer_activations, period="stim", pooling=pooling, n_folds=n_folds
)
infer_cross_stim_acc = crosscontext_decode(
    infer_activations, period="stim", pooling=pooling
)
infer_gen_stim = generalisation_matrix(infer_stim_acc, infer_cross_stim_acc)

infer_r_within = value_decode_within(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix, n_folds=n_folds,
)
infer_r_cross = value_decode_cross(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix,
)
infer_gm_val = value_gen_matrix(infer_r_within, infer_r_cross)

fig, axes = plt.subplots(
    1, 2,
    figsize=(cell_size * n_contexts * 2 + 1, cell_size * n_contexts + 1),
    squeeze=False,
)
plot_generalisation_heatmap(
    axes[0, 0], infer_gen_stim, contexts,
    vmin=0, vmax=1, cmap="seismic",
    colorbar_label="Accuracy",
    title="Stim identity",
)
plot_generalisation_heatmap(
    axes[0, 1], infer_gm_val, contexts,
    vmin=-1, vmax=1, cmap="seismic",
    colorbar_label="Pearson r",
    title="Value — Ridge",
)
fig.suptitle(
    f"Inference decoding  ·  stim period  ·  {pooling} pooling  [plasticity off]",
    y=1.06,
)
plt.tight_layout()
fig.savefig(save_dir / "fig5_decoding_generalisation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Figure 5b: Ridge value generalisation by stimulus group ─────────────
# Each panel trains AND tests on the same stimulus group only.
infer_ridge_gm = {}
for gname, gidx in stim_groups.items():
    act_g = (
        filter_act_dict(infer_activations, np.isin(infer_activations["stimulus"], gidx))
        if gidx is not None
        else infer_activations
    )
    rw = value_decode_within(
        act_g, period="stim", pooling=pooling,
        value_matrix=value_matrix, n_folds=n_folds,
    )
    rc = value_decode_cross(
        act_g, period="stim", pooling=pooling,
        value_matrix=value_matrix,
    )
    infer_ridge_gm[gname] = value_gen_matrix(rw, rc)

fig, axes = plt.subplots(
    1, len(stim_groups),
    figsize=(cell_size * n_contexts * len(stim_groups) + 1, cell_size * n_contexts + 1),
    squeeze=False,
)
for ax, (gname, gm) in zip(axes[0], infer_ridge_gm.items()):
    plot_generalisation_heatmap(
        ax, gm, contexts,
        vmin=-1, vmax=1, cmap="seismic",
        colorbar_label="Pearson r",
        title=f"Ridge — train={gname}, test={gname}",
    )
fig.suptitle(
    f"Inference Ridge value  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
    "Train and test both restricted to the same stimulus group",
    y=1.08,
)
plt.tight_layout()
fig.savefig(save_dir / "fig5b_ridge_value_by_group.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 5c — Ridge value decoding: train=all stimuli, test per group

Same decoder trained on all stimuli in the training context, but evaluated separately
on each stimulus group at test time.  Contrasts with Fig 5b where training is also
restricted to the group.  Cross-context entries for swap stimuli (r ≈ −1) indicate an
identity-weighted solution; positive cross-context r for anchors under the same decoder
would confirm this interpretation.

In [ ]:
# ── Figure 5c: Ridge value — train=all stimuli, test per group ───────────
# Decoder is always trained on all stimuli; each panel tests on a different subset.
# Within-context diagonal uses 5-fold CV (test-group trials held out of training).
_cgd_5c = cross_group_decode(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix,
    train_groups={"all": None},
    test_groups=stim_groups,
    n_folds=n_folds,
)

fig, axes = plt.subplots(
    1, len(stim_groups),
    figsize=(cell_size * n_contexts * len(stim_groups) + 1, cell_size * n_contexts + 1),
    squeeze=False,
)
for ax, (gname, gidx) in zip(axes[0], stim_groups.items()):
    plot_generalisation_heatmap(
        ax, _cgd_5c[("all", gname)], contexts,
        vmin=-1, vmax=1, cmap="seismic",
        colorbar_label="Pearson r",
        title=f"Ridge — train=all, test={gname}",
    )
fig.suptitle(
    f"Inference Ridge value  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
    "Decoder trained on all stimuli, tested on each group separately",
    y=1.08,
)
plt.tight_layout()
fig.savefig(save_dir / "fig5c_ridge_train_all_test_by_group.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 6 — Binary SVM value decoding by stimulus group

In [ ]:
# ── Figure 6: binary SVM (low vs high value) by stimulus group ───────────
# Each panel trains AND tests on the same stimulus group only.
infer_bin_gm = {}
for gname, gidx in stim_groups.items():
    act_g = (
        filter_act_dict(infer_activations, np.isin(infer_activations["stimulus"], gidx))
        if gidx is not None
        else infer_activations
    )
    bw = binary_value_decode_within(
        act_g, period="stim", pooling=pooling,
        value_matrix=value_matrix, n_folds=n_folds,
    )
    bc = binary_value_decode_cross(
        act_g, period="stim", pooling=pooling,
        value_matrix=value_matrix,
    )
    gm = bc.copy()
    np.fill_diagonal(gm, bw)
    infer_bin_gm[gname] = gm

fig, axes = plt.subplots(
    1, len(stim_groups),
    figsize=(cell_size * n_contexts * len(stim_groups) + 1, cell_size * n_contexts + 1),
    squeeze=False,
)
for ax, (gname, gm) in zip(axes[0], infer_bin_gm.items()):
    plot_generalisation_heatmap(
        ax, gm, contexts,
        vmin=0, vmax=1, cmap="seismic",
        colorbar_label="Accuracy",
        title=f"Binary SVM — train={gname}, test={gname}",
    )
fig.suptitle(
    f"Inference binary SVM  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
    "Train and test both restricted to the same stimulus group  "
    "(low (v=0) vs high (v=1), identities pooled within group)",
    y=1.08,
)
plt.tight_layout()
fig.savefig(save_dir / "fig6_binary_svm_decoding.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 6c — Binary SVM: train=all stimuli, test per group

Same classifier trained on all stimuli; tested on anchor-only or swap-only separately.
Parallel to Fig 5c but for binary SVM.  Contrasts with Fig 6 where training is also
restricted to the group.

In [ ]:
# ── Figure 6c: binary SVM — train=all stimuli, test per group ────────────
# Classifier trained on all stimuli; each panel tests on a different subset.
# Within-context diagonal uses 5-fold CV (test-group trials held out of training).
_cgd_bin_6c = cross_group_binary_decode(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix,
    train_groups={"all": None},
    test_groups=stim_groups,
    n_folds=n_folds,
)

fig, axes = plt.subplots(
    1, len(stim_groups),
    figsize=(cell_size * n_contexts * len(stim_groups) + 1, cell_size * n_contexts + 1),
    squeeze=False,
)
for ax, (gname, gidx) in zip(axes[0], stim_groups.items()):
    plot_generalisation_heatmap(
        ax, _cgd_bin_6c[("all", gname)], contexts,
        vmin=0, vmax=1, cmap="seismic",
        colorbar_label="Accuracy",
        title=f"Binary SVM — train=all, test={gname}",
    )
fig.suptitle(
    f"Inference binary SVM  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
    "Classifier trained on all stimuli, tested on each group separately",
    y=1.08,
)
plt.tight_layout()
fig.savefig(save_dir / "fig6c_binary_svm_train_all_test_by_group.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 7 — Unit selectivity and value preference

In [ ]:
# ── Figure 7: unit selectivity and value preference ──────────────────────
si_threshold    = 0.1
value_threshold = 0.25

tuning          = compute_unit_tuning(infer_activations, period="stim")
props, si_vals  = preferred_value_proportions(
    tuning, value_matrix, si_threshold=si_threshold, value_threshold=value_threshold
)
n_units    = tuning.shape[0]
n_sel      = int((si_vals >= si_threshold).sum())

# sort units by preferred stimulus (mean across contexts), then by SI
mean_ctx   = np.nanmean(tuning, axis=2)                    # (n_units, n_stimuli)
pref_stim  = np.nanargmax(mean_ctx, axis=1)
sort_idx   = np.lexsort((si_vals, pref_stim))              # primary: pref_stim, secondary: si

tuning_sorted = mean_ctx[sort_idx]
u_mean  = np.nanmean(tuning_sorted, axis=1, keepdims=True)
u_std   = np.nanstd(tuning_sorted,  axis=1, keepdims=True)
u_std[u_std == 0] = 1.0
tuning_z = (tuning_sorted - u_mean) / u_std

fig = plt.figure(figsize=(15, 5.5))
gs  = fig.add_gridspec(1, 3, wspace=0.35)

# ── heatmap ───────────────────────────────────────────────────────────────
ax_h = fig.add_subplot(gs[0])
im   = ax_h.imshow(tuning_z, aspect="auto", cmap="RdBu_r",
                   vmin=-2.5, vmax=2.5, interpolation="nearest")
ax_h.set_xticks(range(n_stimuli))
ax_h.set_xticklabels(stimuli, fontsize=7)
ax_h.set_xlabel("Stimulus")
ax_h.set_ylabel(f"Unit (sorted by preferred stimulus, n={n_units})")
ax_h.set_title("Stimulus tuning  (z-scored,\nmean across contexts)")
plt.colorbar(im, ax=ax_h, shrink=0.6)

# ── SI distribution ───────────────────────────────────────────────────────
ax_si = fig.add_subplot(gs[1])
ax_si.hist(si_vals, bins=25, color="steelblue", edgecolor="white", alpha=0.85)
ax_si.axvline(si_threshold, color="crimson", lw=1.5, linestyle="--",
              label=f"threshold = {si_threshold}")
ax_si.set_xlabel("Selectivity index")
ax_si.set_ylabel("Number of units")
ax_si.set_title(f"SI distribution\n({n_sel}/{n_units} units selective)")
ax_si.legend(fontsize=8)

# ── value preference proportions ─────────────────────────────────────────
ax_p = fig.add_subplot(gs[2])
_grp_colors = {"low": "#4C72B0", "mid": "#8C8C8C", "high": "#DD8452"}
x_ctx   = np.arange(n_contexts)
bottoms = np.zeros(n_contexts)
for gname, gc in _grp_colors.items():
    vals = np.array([props[ci][f"frac_{gname}"] * 100 for ci in range(n_contexts)])
    ax_p.bar(x_ctx, vals, width=0.5, bottom=bottoms,
             color=gc, alpha=0.85, label=gname)
    for ci in range(n_contexts):
        v = vals[ci]
        if not np.isnan(v) and v > 4:
            ax_p.text(ci, bottoms[ci] + v / 2, f"{v:.0f}%",
                      ha="center", va="center", fontsize=9,
                      color="white", fontweight="bold")
    bottoms += np.nan_to_num(vals)

for ci in range(n_contexts):
    ax_p.text(ci, -7, f"n={props[ci]['n_selective']}",
              ha="center", va="top", fontsize=8, color="0.35")

ax_p.set_xticks(x_ctx)
ax_p.set_xticklabels(contexts)
ax_p.set_ylim(0, 108)
ax_p.set_ylabel("% of selective units")
ax_p.set_title(
    f"Value preference of selective units\n(SI ≥ {si_threshold}, "
    f"low ≤ {value_threshold}, high ≥ {1 - value_threshold:.2f})"
)
ax_p.legend(fontsize=8, loc="upper right")

fig.suptitle(
    "Unit selectivity analysis  [inference activations, stim period, plasticity off]",
    y=1.02,
)
plt.tight_layout()
fig.savefig(save_dir / "fig7_unit_selectivity.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 8 — Time-resolved value decoding

In [ ]:
# ── Figure 8: time-resolved binary value decoding ───────────────────────
_n_iti_pre = 3

acc_within_tr, acc_cross_tr, ep = time_resolved_decode(
    infer_activations, value_matrix,
    n_iti_pre=_n_iti_pre,
    value_threshold=value_threshold,
    n_folds=n_folds,
)

t_axis  = np.arange(ep["total"])
t_labels = (
    [f"ITI-{_n_iti_pre - i}" for i in range(_n_iti_pre)]
    + [f"S{i + 1}" for i in range(ep["reward_onset"] - ep["stim_onset"])]
    + [f"R{i + 1}" for i in range(ep["total"] - ep["reward_onset"])]
)

_epoch_spans = [
    (0,                   ep["stim_onset"],   "lightgray", "ITI"),
    (ep["stim_onset"],    ep["reward_onset"], "#4C72B0",   "Stim"),
    (ep["reward_onset"],  ep["total"],        "#DD8452",   "Reward"),
]

_CTX_PALETTE = ["#5C6BC0", "#FFA726", "#66BB6A", "#EF5350", "#26C6DA"]
_ctx_colors_tr = [_CTX_PALETTE[ci % len(_CTX_PALETTE)] for ci in range(n_contexts)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax in axes:
    for es, ee, ec, _ in _epoch_spans:
        ax.axvspan(es - 0.5, ee - 0.5, alpha=0.12, color=ec, zorder=0)
    ax.axhline(0.5, color="gray", lw=0.8, linestyle="--", alpha=0.7, label="chance")
    ax.set_xticks(t_axis)
    ax.set_xticklabels(t_labels, fontsize=7, rotation=45, ha="right")
    ax.set_ylim(0.3, 1.05)
    ax.set_xlabel("Aligned timestep")

# ── within-context ────────────────────────────────────────────────────────
ax = axes[0]
for ci in range(n_contexts):
    ax.plot(t_axis, acc_within_tr[ci], color=_ctx_colors_tr[ci],
            lw=2, marker="o", ms=4, label=contexts[ci])
ax.set_ylabel("Decoding accuracy")
ax.set_title("Within-context  (k-fold CV)")

from matplotlib.patches import Patch
from matplotlib.lines import Line2D as _Line2D
epoch_handles = [Patch(facecolor=ec, alpha=0.5, label=el)
                 for _, _, ec, el in _epoch_spans]
ctx_handles   = [_Line2D([0], [0], color=_ctx_colors_tr[ci], lw=2, label=contexts[ci])
                 for ci in range(n_contexts)]
axes[0].legend(handles=epoch_handles + ctx_handles, fontsize=8, ncol=2)

# ── cross-context ─────────────────────────────────────────────────────────
ax = axes[1]
for c_train in range(n_contexts):
    for c_test in range(n_contexts):
        if c_train == c_test:
            continue
        ax.plot(t_axis, acc_cross_tr[c_train, c_test],
                color=_ctx_colors_tr[c_train], lw=2, linestyle="--",
                marker="o", ms=4, label=f"{contexts[c_train]}→{contexts[c_test]}")
ax.set_title("Cross-context  (train → test)")
ax.legend(fontsize=8)

fig.suptitle(
    "Time-resolved binary value decoding  "
    f"[inference, {_n_iti_pre} ITI + stim + reward window, plasticity off]",
    y=1.02,
)
plt.tight_layout()
fig.savefig(save_dir / "fig8_time_resolved_decoding.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 9 — Identity-partialled value decoding

Stimulus identity (one-hot) is regressed out of the hidden states within each
CV fold before fitting the Ridge / SVM decoder.  Any remaining decodable signal
is orthogonal to what identity alone explains.  If cross-context r is still
negative after partialling, there is no separable value axis in the
representation.

In [ ]:
# ── Figure 9: identity-partialled Ridge + binary SVM ─────────────────────
r_within_p = value_decode_within_partialled(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix, n_folds=n_folds,
)
r_cross_p  = value_decode_cross_partialled(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix,
)
gm_val_p   = value_gen_matrix(r_within_p, r_cross_p)

bw_p = binary_value_decode_within_partialled(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix, n_folds=n_folds,
)
bc_p = binary_value_decode_cross_partialled(
    infer_activations, period="stim", pooling=pooling,
    value_matrix=value_matrix,
)
gm_bin_p = bc_p.copy()
np.fill_diagonal(gm_bin_p, bw_p)

print("Ridge (partialled)  within r:", r_within_p)
print("Ridge (partialled)  cross  r:", r_cross_p)
print("SVM   (partialled)  within acc:", bw_p)
print("SVM   (partialled)  cross  acc:", bc_p)

fig, axes = plt.subplots(
    1, 2,
    figsize=(cell_size * n_contexts * 2 + 1, cell_size * n_contexts + 1),
    squeeze=False,
)
plot_generalisation_heatmap(
    axes[0, 0], gm_val_p, contexts,
    vmin=-1, vmax=1, cmap="seismic",
    colorbar_label="Pearson r",
    title="Ridge (identity-partialled)",
)
plot_generalisation_heatmap(
    axes[0, 1], gm_bin_p, contexts,
    vmin=0, vmax=1, cmap="seismic",
    colorbar_label="Accuracy",
    title="Binary SVM (identity-partialled)",
)
fig.suptitle(
    f"Identity-partialled value decoding  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
    "One-hot identity regressed from hidden states within each fold before decoding",
    y=1.1,
)
plt.tight_layout()
fig.savefig(save_dir / "fig9_identity_partialled_decoding.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 10 — Representational Similarity Analysis (RSA)

Computes the neural RDM (pairwise 1 − Pearson correlation between mean hidden
states per stimulus × context condition) and compares it against two model RDMs:

- **Identity model**: dissimilarity = 0 if same stimulus (any context), 1 if different
- **Value model**: dissimilarity = |value_i − value_j|

Spearman r between the upper triangle of the neural RDM and each model RDM tells
us which model better predicts the geometry of the representation.  In the swap
task, the two models make opposite predictions for cross-context pairs
(same stimulus, different value), making them dissociable.

In [ ]:
# ── Figure 10: RSA across trial periods ──────────────────────────────────────
from cxval.vis import add_colorbar

_n_iti_pre     = 3    # ITI timesteps to use (matches Fig 8)
_periods       = ["iti", "stim", "reward"]
_period_labels = {"iti": f"ITI (last {_n_iti_pre} ts)", "stim": "Stim", "reward": "Reward"}

# model RDMs are period-independent — compute once
_, _conditions = compute_neural_rdm(infer_activations, period="stim")
id_rdm, val_rdm = compute_model_rdms(_conditions, value_matrix)
cond_labels = [f"s{si},c{ci}" for si, ci in _conditions]

# neural RDM + RSA r for each period
_neural_rdms = {}
_rsa_results = {}
for p in _periods:
    rdm, _ = compute_neural_rdm(infer_activations, period=p, n_iti_pre=_n_iti_pre)
    _neural_rdms[p] = rdm
    _rsa_results[p] = rsa_compare(rdm, {"identity": id_rdm, "value": val_rdm})
    print(f"{_period_labels[p]:25s}  identity r={_rsa_results[p]['identity']:+.3f}  "
          f"value r={_rsa_results[p]['value']:+.3f}")

# ── Fig 10a: neural RDMs (one per period) + model RDMs ───────────────────────
n_panels = len(_periods) + 2
fig, axes = plt.subplots(1, n_panels, figsize=(3.5 * n_panels, 4))

for ax, p in zip(axes[:len(_periods)], _periods):
    rdm = _neural_rdms[p]
    vm  = max(np.nanmax(rdm), 0.01)
    im  = ax.imshow(rdm, cmap="viridis", vmin=0, vmax=vm, aspect="equal")
    ax.set_xticks(range(len(cond_labels)))
    ax.set_xticklabels(cond_labels, fontsize=7, rotation=45, ha="right")
    ax.set_yticks(range(len(cond_labels)))
    ax.set_yticklabels(cond_labels, fontsize=7)
    ax.set_title(f"Neural RDM\n{_period_labels[p]}", fontsize=9)
    add_colorbar(ax, im, label="1 − Pearson r")

for ax, rdm, title in zip(axes[len(_periods):],
                           [id_rdm, val_rdm],
                           ["Identity model RDM", "Value model RDM"]):
    vm = max(np.nanmax(rdm), 0.01)
    im = ax.imshow(rdm, cmap="viridis", vmin=0, vmax=vm, aspect="equal")
    ax.set_xticks(range(len(cond_labels)))
    ax.set_xticklabels(cond_labels, fontsize=7, rotation=45, ha="right")
    ax.set_yticks(range(len(cond_labels)))
    ax.set_yticklabels(cond_labels, fontsize=7)
    ax.set_title(title, fontsize=9)
    add_colorbar(ax, im)

fig.suptitle(
    "RSA — neural RDMs and model RDMs  [inference, plasticity off]",
    y=1.02,
)
plt.tight_layout()
fig.savefig(save_dir / "fig10_rsa_rdms.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Fig 10b: RSA summary — grouped bars by period ────────────────────────────
x      = np.arange(len(_periods))
width  = 0.35
colors = {"identity": "#4C72B0", "value": "#DD8452"}

fig, ax = plt.subplots(figsize=(5, 3.5))
for i, model in enumerate(["identity", "value"]):
    vals = [_rsa_results[p][model] for p in _periods]
    bars = ax.bar(x + (i - 0.5) * width, vals, width,
                  color=colors[model], label=model, alpha=0.85)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            yoff = 0.03 if val >= 0 else -0.08
            va   = "bottom" if val >= 0 else "top"
            ax.text(bar.get_x() + bar.get_width() / 2, val + yoff,
                    f"{val:.2f}", ha="center", va=va, fontsize=8)

ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([_period_labels[p] for p in _periods])
ax.set_ylim(-1.1, 1.1)
ax.set_ylabel("Spearman r  (neural vs model RDM)")
ax.set_title("RSA across trial periods")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(save_dir / "fig10b_rsa_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 11 — Held-out stimulus decoding (anchor → swap)

Train Ridge on anchor stimuli (constant value across contexts), test on swap
stimuli (value reverses across contexts).  The decoder can only generalise if
hidden states for swap stimuli at test time are organised along the same axis the
decoder found in the anchor stimuli.

Skipped automatically if the task has no anchor or no swap stimuli.

In [ ]:
# ── Figure 11: held-out stimulus decoding (anchor → swap) ─────────────────
_swap_idx   = [i for g in stim_group_info if "swap"   in g["name"] for i in g["indices"]]
_anchor_idx = [i for g in stim_group_info if "anchor" in g["name"] for i in g["indices"]]

if not _swap_idx or not _anchor_idx:
    print(
        "Held-out decoding requires both anchor and swap stimuli — skipping Fig 11.\n"
        f"  swap indices  : {_swap_idx}\n"
        f"  anchor indices: {_anchor_idx}"
    )
else:
    r_held_out = held_out_stim_decode(
        infer_activations, period="stim", pooling=pooling,
        value_matrix=value_matrix,
        train_stim_idx=_anchor_idx,
        test_stim_idx=_swap_idx,
    )
    print("Held-out r matrix (train context × test context):")
    print(r_held_out)

    fig, ax = plt.subplots(
        figsize=(cell_size * n_contexts + 0.5, cell_size * n_contexts + 0.5)
    )
    plot_generalisation_heatmap(
        ax, r_held_out, contexts,
        vmin=-1, vmax=1, cmap="seismic",
        colorbar_label="Pearson r",
        title="Held-out: train anchor, test swap",
        xlabel="Test context", ylabel="Train context",
    )
    anchor_stim_names = [stimuli[i] for i in _anchor_idx]
    swap_stim_names   = [stimuli[i] for i in _swap_idx]
    fig.suptitle(
        f"Held-out stimulus Ridge decoding  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
        f"Train stimuli: {anchor_stim_names}  ·  Test stimuli: {swap_stim_names}",
        y=1.1,
    )
    plt.tight_layout()
    fig.savefig(save_dir / "fig11_held_out_stim_decoding.png", dpi=150, bbox_inches="tight")
    plt.show()

## Figure 12 — Decoder axis comparison (all-stim decoder, tested on anchor vs swap)

Two analyses to verify whether the all-stim decoder is finding a value axis or an identity axis.

**12a — 1×2 cross-group decoding.**  Train Ridge on all stimuli; test on anchor-only or
swap-only in every context, using 5-fold CV for within-context entries (test-group trials
are held out of training in each fold).  If the decoder found an identity solution, it
should generalise to anchors but fail on swaps cross-context (r ≈ −1).  If it found a
value axis, both test groups should generalise.

**12b — Cosine similarity of decoder weight vectors.**  Anchor-only and all-stim decoders
are fit using a shared StandardScaler so their weight vectors live in the same coordinate
system.  Cosine ≈ 0 means the anchor decoder found a direction orthogonal to the all-stim
decoder; cosine ≈ ±1 means they found the same axis.

Skipped if anchor or swap stimuli are absent from the task.

In [ ]:
# ── Figure 12: decoder axis comparison ───────────────────────────────────────
_swap_idx   = [i for g in stim_group_info if "swap"   in g["name"] for i in g["indices"]]
_anchor_idx = [i for g in stim_group_info if "anchor" in g["name"] for i in g["indices"]]

if not _swap_idx or not _anchor_idx:
    print("Fig 12 requires both anchor and swap stimuli — skipping.")
else:
    # ── 12a: train=all, test={anchor, swap} with 5-fold CV on diagonal ───────
    _train_groups = {"all": None}
    _test_groups  = {"anchor": _anchor_idx, "swap": _swap_idx}

    _cgd = cross_group_decode(
        infer_activations, period="stim", pooling=pooling,
        value_matrix=value_matrix,
        train_groups=_train_groups,
        test_groups=_test_groups,
        n_folds=n_folds,
    )

    _test_names = list(_test_groups.keys())

    fig, axes = plt.subplots(
        1, len(_test_names),
        figsize=(cell_size * n_contexts * len(_test_names) + 1.5,
                 cell_size * n_contexts + 1.5),
        squeeze=False,
    )
    for ci, te_name in enumerate(_test_names):
        r_mat = _cgd[("all", te_name)]
        plot_generalisation_heatmap(
            axes[0, ci], r_mat, contexts,
            vmin=-1, vmax=1, cmap="seismic",
            colorbar_label="Pearson r",
            title=f"train=all → test={te_name}",
        )
    axes[0, 0].set_ylabel("Train context", fontsize=8)

    fig.suptitle(
        f"Cross-group Ridge decoding  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
        f"anchor stim: {[stimuli[i] for i in _anchor_idx]}  "
        f"swap stim: {[stimuli[i] for i in _swap_idx]}",
        y=1.06,
    )
    plt.tight_layout()
    fig.savefig(save_dir / "fig12a_cross_group_decode.png", dpi=150, bbox_inches="tight")
    plt.show()

    # print summary of key cross-context entries
    print("\nKey cross-context r values (train=all):")
    for te_name in _test_names:
        r_mat = _cgd[("all", te_name)]
        offdiag = [r_mat[i, j] for i in range(n_contexts)
                   for j in range(n_contexts) if i != j and not np.isnan(r_mat[i, j])]
        mean_od = np.mean(offdiag) if offdiag else np.nan
        diag    = [r_mat[i, i] for i in range(n_contexts) if not np.isnan(r_mat[i, i])]
        mean_d  = np.mean(diag) if diag else np.nan
        print(f"  test={te_name:6s}  within r = {mean_d:+.3f}  cross r = {mean_od:+.3f}")

    # ── 12b: decoder weight cosine similarity ─────────────────────────────────
    cos_sim, w_anchor, w_all = decoder_axis_similarity(
        infer_activations, period="stim", pooling=pooling,
        value_matrix=value_matrix,
        stim_idx_a=_anchor_idx,
        stim_idx_b=None,           # all stimuli
    )

    print(f"\nDecoder weight cosine similarity (anchor vs all-stim):")
    for ci in range(n_contexts):
        print(f"  {contexts[ci]}: cos = {cos_sim[ci]:+.3f}")

    _CTX_PALETTE = ["#5C6BC0", "#FFA726", "#66BB6A", "#EF5350", "#26C6DA"]
    _ctx_col = [_CTX_PALETTE[ci % len(_CTX_PALETTE)] for ci in range(n_contexts)]

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

    # cosine similarity bar
    ax = axes[0]
    bars = ax.bar(contexts, cos_sim, color=_ctx_col, width=0.5, alpha=0.85)
    ax.axhline(0,  color="black", lw=0.8)
    ax.axhline( 1, color="gray",  lw=0.6, linestyle="--", alpha=0.5)
    ax.axhline(-1, color="gray",  lw=0.6, linestyle="--", alpha=0.5)
    ax.set_ylim(-1.1, 1.1)
    ax.set_ylabel("Cosine similarity")
    ax.set_title("Decoder weight similarity\n(anchor-only vs all-stim Ridge)")
    for bar, val in zip(bars, cos_sim):
        if not np.isnan(val):
            yoff = 0.04 if val >= 0 else -0.1
            ax.text(bar.get_x() + bar.get_width() / 2, val + yoff,
                    f"{val:+.3f}", ha="center",
                    va="bottom" if val >= 0 else "top", fontsize=10)

    # weight vector norms (sanity check that both decoders found something)
    ax = axes[1]
    x_ctx = np.arange(n_contexts)
    width = 0.35
    for i, (w, label) in enumerate([(w_anchor, "anchor"), (w_all, "all-stim")]):
        norms = [np.linalg.norm(w[ci]) if not np.isnan(w[ci]).any() else np.nan
                 for ci in range(n_contexts)]
        ax.bar(x_ctx + (i - 0.5) * width, norms, width, label=label, alpha=0.85)
    ax.set_xticks(x_ctx)
    ax.set_xticklabels(contexts)
    ax.set_ylabel("||w||  (weight vector norm)")
    ax.set_title("Decoder weight norms\n(shared scaler)")
    ax.legend(fontsize=9)

    fig.suptitle(
        "Decoder axis comparison  [inference, stim period, plasticity off]", y=1.03
    )
    plt.tight_layout()
    fig.savefig(save_dir / "fig12b_decoder_axis_similarity.png", dpi=150, bbox_inches="tight")
    plt.show()